# IS 4487 Assignment 11: Predicting Airbnb Prices with Regression

In this assignment, you will:
- Load the Airbnb dataset you cleaned and transformed in Assignment 7
- Build a linear regression model to predict listing price
- Interpret which features most affect price
- Try to improve your model using only the most impactful predictors
- Practice explaining your findings to a business audience like a host, pricing strategist, or city partner

## Why This Matters

Pricing is one of the most important levers for hosts and Airbnb’s business teams. Understanding what drives price — and being able to predict it accurately — helps improve search results, revenue management, and guest satisfaction.

This assignment gives you hands-on practice turning a cleaned dataset into a predictive model. You’ll focus not just on code, but on what the results mean and how you’d communicate them to stakeholders.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_11_regression.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



## Original Source: Dataset Description

The dataset you'll be using is a **detailed Airbnb listing file**, available from [Inside Airbnb](https://insideairbnb.com/get-the-data/).

Each row represents one property listing. The columns include:

- **Host attributes** (e.g., host ID, host name, host response time)
- **Listing details** (e.g., price, room type, minimum nights, availability)
- **Location data** (e.g., neighborhood, latitude/longitude)
- **Property characteristics** (e.g., number of bedrooms, amenities, accommodates)
- **Calendar/booking variables** (e.g., last review date, number of reviews)

The schema is consistent across cities, so you can expect similar columns regardless of the location you choose.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


## 1. Load Your Transformed Airbnb Dataset

**Business framing:**  
Before building any models, we must start with clean, prepared data. In Assignment 7, you exported a cleaned version of your Airbnb dataset. You’ll now import that file for analysis.

### Do the following:
- Import your CSV file called `cleaned_airbnb_data_7.csv`.   (Note: If you had significant errors with assignment 7, you can use the file named "airbnb_listings.csv" in the DataSets folder on GitHub as a backup starting point.)
- Use `pandas` to load and preview the dataset


In [12]:
# Add code here 🔧
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/main/DataSets/airbnb_listings.csv')
display(df.head())
display(df.info())

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,host_url,host_name,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,2992450,https://www.airbnb.com/rooms/2992450,20250804133828,2025-08-04,city scrape,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,NaN,https://www.airbnb.com/users/show/4621559,Kenneth,...,4.56,3.22,3.67,NaN,0,1,1,0,0,0.07
1,3820211,https://www.airbnb.com/rooms/3820211,20250804133828,2025-08-04,city scrape,Funky Urban Gem: Prime Central Location - Park...,Step into the charming and comfy 1BR/1BA apart...,Overview<br /><br />The lovely apartment is lo...,https://www.airbnb.com/users/show/19648678,Terra,...,4.81,4.81,4.77,NaN,0,4,4,0,0,2.32
2,5651579,https://www.airbnb.com/rooms/5651579,20250804133828,2025-08-04,city scrape,Large studio apt by Capital Center & ESP@,"Spacious studio with hardwood floors, fully eq...",The neighborhood is very eclectic. We have a v...,https://www.airbnb.com/users/show/29288920,Gregg,...,4.88,4.76,4.64,NaN,0,2,1,1,0,2.97
3,6623339,https://www.airbnb.com/rooms/6623339,20250804133828,2025-08-04,city scrape,Bright & Cozy City Stay · Top Location + Parking!,Step into the charming and comfy 1BR/1BA apart...,Overview<br /><br />The lovely apartment is lo...,https://www.airbnb.com/users/show/19648678,Terra,...,4.70,4.80,4.72,NaN,0,4,4,0,0,2.68
4,9005989,https://www.airbnb.com/rooms/9005989,20250804133828,2025-08-04,city scrape,"Studio in The heart of Center SQ, in Albany NY",(21 years of age or older ONLY) NON- SMOKING.....,"There are many shops, restaurants, bars, museu...",https://www.airbnb.com/users/show/17766924,Sugey,...,4.93,4.87,4.77,NaN,0,1,1,0,0,5.67


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 459 entries, 0 to 458
Data columns (total 77 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            459 non-null    int64  
 1   listing_url                                   459 non-null    object 
 2   scrape_id                                     459 non-null    int64  
 3   last_scraped                                  459 non-null    object 
 4   source                                        459 non-null    object 
 5   name                                          459 non-null    object 
 6   description                                   449 non-null    object 
 7   neighborhood_overview                         196 non-null    object 
 8   host_url                                      459 non-null    object 
 9   host_name                                     459 non-null    obj

None

### Reflection:
1. What business information does the dataset include?
2. How many rows and columns are present?

### ✍️ Your Response: 🔧
1. I noticed the dataset has super useful business details like listing prices, location, review ratings, host response rates, and property characteristics like room types.

2. Based on the outputs, there are 459 rows and 77 columns in this dataset.

## 2. Drop Columns Not Useful for Modeling

**Business framing:**  
Some columns — like post IDs or text — may not help us predict price and could add noise or bias.

### Do the following:
- Drop columns like `post_id`, `title`, `descr`, `details`, and `address` if they’re still in your dataset

In [13]:
# Add code here 🔧
exclude_cols = ['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'host_url', 'host_name', 'host_since', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_verifications', 'host_has_profile_pic', 'calendar_last_scraped', 'first_review', 'last_review', 'license']
df_cleaned = df.drop(columns=exclude_cols, errors='ignore')
df_numeric = df_cleaned.select_dtypes(include=['int64', 'float64'])
display(df_numeric.head())

,host_listings_count,host_total_listings_count,latitude,longitude,accommodates,bathrooms,bedrooms,beds,price,minimum_nights,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,1,5,42.65789,-73.75370,4,1.0,2.0,2.0,70.0,28,...,4.22,4.56,3.22,3.67,0,1,1,0,0,0.07
1,4,6,42.65222,-73.76724,2,1.0,1.0,1.0,104.0,2,...,4.85,4.81,4.81,4.77,0,4,4,0,0,2.32
2,2,2,42.64615,-73.75966,2,1.0,0.0,1.0,75.0,2,...,4.81,4.88,4.76,4.64,0,2,1,1,0,2.97
3,4,6,42.65222,-73.76724,2,1.0,1.0,1.0,101.0,2,...,4.83,4.70,4.80,4.72,0,4,4,0,0,2.68
4,1,1,42.65559,-73.76506,4,1.0,1.0,2.0,110.0,1,...,4.95,4.93,4.87,4.77,0,1,1,0,0,5.67


### Reflection:
1. What columns did you drop, and why?
2. What risks might occur if you included them in your model?

### ✍️ Your Response: 🔧
1. I dropped text and high-cardinality ID columns like listing URLs, host names, and descriptions because they contain non-numeric data that linear regression models cannot process.

2. Including them could cause model errors, slow down training, or introduce meaningless noise and overfitting to specific text patterns rather than generalizable features.

## 3. Explore Relationships Between Numeric Features

**Business framing:**  
Understanding how features relate to each other — and to the target — helps guide feature selection and modeling.

### Do the following:
- Generate a correlation matrix
- Identify which variables are strongly related to `price`

In [14]:
# Add code here 🔧
correlations = df_numeric.corr()['price'].sort_values(ascending=False)
display(correlations)

,price
price,1.000000
accommodates,0.579588
beds,0.547032
bedrooms,0.499286
bathrooms,0.468030
estimated_revenue_l365d,0.249488
maximum_maximum_nights,0.122872
minimum_maximum_nights,0.112166
maximum_nights_avg_ntm,0.111271
availability_30,0.108409


### Reflection:
1. Which variables had the strongest positive or negative correlation with price?
2. Which variables might be useful predictors?

### ✍️ Your Response: 🔧
1. I noticed that accommodates (0.58), beds (0.55), and bedrooms (0.50) have the strongest positive correlations with price. Review scores for communication had the strongest negative correlation (-0.13).

2. The size-related variables like accommodates, beds, bedrooms, and bathrooms will definitely be the most useful predictors of price.

## 4. Define Features and Target Variable

**Business framing:**  
To build a regression model, you need to define what you’re predicting (target) and what you’re using to make that prediction (features).

### Do the following:
- Set `price` as your target variable
- Remove `price` from your predictors

In [15]:
# Add code here 🔧
df_model = df_numeric.dropna()
y = df_model['price']
X = df_model.drop(columns=['price'])
display(X.head())

,host_listings_count,host_total_listings_count,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,maximum_nights,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,1,5,42.65789,-73.75370,4,1.0,2.0,2.0,28,1125,...,4.22,4.56,3.22,3.67,0,1,1,0,0,0.07
1,4,6,42.65222,-73.76724,2,1.0,1.0,1.0,2,1125,...,4.85,4.81,4.81,4.77,0,4,4,0,0,2.32
2,2,2,42.64615,-73.75966,2,1.0,0.0,1.0,2,45,...,4.81,4.88,4.76,4.64,0,2,1,1,0,2.97
3,4,6,42.65222,-73.76724,2,1.0,1.0,1.0,2,1125,...,4.83,4.70,4.80,4.72,0,4,4,0,0,2.68
4,1,1,42.65559,-73.76506,4,1.0,1.0,2.0,1,1125,...,4.95,4.93,4.87,4.77,0,1,1,0,0,5.67


### Reflection:
1. What features are you using?
2. Why is this a regression problem and not a classification problem?

### ✍️ Your Response: 🔧
1. I am using all available numeric columns such as accommodates, bedrooms, beds, bathrooms, minimum_nights, and review scores as our predictive features.

2. This is a regression problem because we are predicting price, which is a continuous numeric value, rather than predicting a categorical group or class.

## 5. Split Data into Training and Testing Sets

### Business framing:
Splitting your data lets you train a model and test how well it performs on new, unseen data.

### Do the following:
- Use `train_test_split()` to split into 80% training, 20% testing



In [16]:
# Add code here 🔧
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

(292, 40) (74, 40)


## 6. Fit a Linear Regression Model

### Business framing:
Linear regression helps you quantify the impact of each feature on price and make predictions for new listings.

### Do the following:
- Fit a linear regression model to your training data
- Use it to predict prices for the test set



In [17]:
# Add code here 🔧
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

## 7. Evaluate Model Performance

### Business framing:  
A good model should make accurate predictions. We’ll use Mean Squared Error (MSE) and R² to evaluate how close our predictions were to the actual prices.

### Do the following:
- Print MSE and R² score for your model

In [18]:
# Add code here 🔧
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('MSE:', mse)
print('R2:', r2)

MSE: 14803.245037774737
R2: 0.2340652754983975


### Reflection:
1. What is your R² score? How well does your model explain price variation?
2. Is your MSE large or small? What could you do to improve it?

### ✍️ Your Response: 🔧
1. The baseline model's R2 score is approximately 0.234 (or 23.4%). This means that about 23.4% of the variance in Airbnb listing prices can be explained by the numeric features included in our model. The remaining 76.6% of the variance is unexplained, indicating that there is room for improvement.

2. The Mean Squared Error (MSE) is 14,803.25, which is relatively large. Taking the square root gives a Root Mean Squared Error (RMSE) of roughly 121.67, meaning our price predictions are off by an average of about 121 dollars. To improve this, we could engineer more advanced features, include non-numeric features (like room types or neighborhood zones) using one-hot encoding, or utilize more complex models like Random Forests or Gradient Boosting.

## 8. Interpret Model Coefficients

### Business framing:
The regression coefficients tell you how each feature impacts price. This can help Airbnb guide hosts and partners.

### Do the following:
- Create a table showing feature names and regression coefficients
- Sort the table so that the most impactful features are at the top

In [19]:
# Add code here 🔧
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': model.coef_})
coef_df['Absolute_Coefficient'] = coef_df['Coefficient'].abs()
coef_df_sorted = coef_df.sort_values(by='Absolute_Coefficient', ascending=False)
display(coef_df_sorted)

,Feature,Coefficient,Absolute_Coefficient
2,latitude,-311.257265,311.257265
3,longitude,-200.119452,200.119452
33,review_scores_value,38.765381,38.765381
5,bathrooms,33.876175,33.876175
31,review_scores_communication,-30.853858,30.853858
11,maximum_minimum_nights,-20.408640,20.408640
28,review_scores_accuracy,20.275709,20.275709
14,minimum_nights_avg_ntm,17.989741,17.989741
4,accommodates,16.112832,16.112832
30,review_scores_checkin,-14.299140,14.299140


### Reflection:
1. Which features increased price the most?
2. Were any surprisingly negative?
3. What business insight could you draw from this?

### ✍️ Your Response: 🔧
1. Based on absolute coefficients, geographical coordinates (latitude and longitude) have the largest mathematical impact, which highlights how crucial location is. Aside from location, high review scores for value (review_scores_value at +38.77) and the number of bathrooms (+33.88) increased the price predictions the most.

2. Interestingly, some review ratings like review_scores_communication (-30.85) and review_scores_rating (-11.92) had negative coefficients. This is somewhat counterintuitive, but it might be due to multicollinearity (since multiple review score metrics are highly correlated with each other, causing coefficients to behave erratically).

3. The key business insight is that size and perceived value drive price. Hosts should focus on maximizing their value score and highlighting physical features like the number of bathrooms in their listings to justify higher price points.

## 9. Try to Improve the Linear Regression Model

### Business framing:
The first version of your model included all available features — but not all features are equally useful. Removing weak or noisy predictors can often improve performance and interpretation.

### Do the following:
1. Choose your top 3–5 features with the strongest absolute coefficients
2. Rebuild the regression model using just those features
3. Compare MSE and R² between the baseline and refined model

In [20]:
# Add code here 🔧
top_features = ['accommodates', 'bedrooms', 'beds', 'bathrooms']
X_refined = X[top_features]
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_refined, y, test_size=0.2, random_state=42)
model_r = LinearRegression()
model_r.fit(X_train_r, y_train_r)
y_pred_r = model_r.predict(X_test_r)
mse_r = mean_squared_error(y_test_r, y_pred_r)
r2_r = r2_score(y_test_r, y_pred_r)
print('Refined MSE:', mse_r)
print('Refined R2:', r2_r)

Refined MSE: 16671.76787136568
Refined R2: 0.13738603266215466


### Reflection:
1. What features did you keep in the refined model, and why?
2. Did model performance improve? Why or why not?
3. Which model would you recommend to stakeholders?
4. How does this relate to your customized learning outcome you created in canvas?

### ✍️ Your Response: 🔧
1. I kept accommodates, bedrooms, beds, and bathrooms because these size-related variables showed the strongest initial correlations with price and are very easy for a potential host to understand and modify.

2. No, model performance did not improve. The R2 fell from 0.234 to 0.137, and the MSE increased from 14,803.25 to 16,671.77. This is because dropping location variables (latitude, longitude) and quality metrics (review_scores_value) stripped away important predictive information from the model.

3. I would recommend the baseline model over the highly simplified model because it explains significantly more of the price variation. However, I would advise stakeholders that we should continue developing the model to include categorical features (like room type) to improve predictions further.

4. This directly relates to my customized learning outcome of understanding how to balance model complexity against predictive accuracy, demonstrating that over-simplifying a model can cost crucial predictive power.

## 10. Reflect and Recommend

### Business framing:  
Ultimately, the value of your model comes from how well it can guide business decisions. Use your results to make real-world recommendations.

### Reflection:
1. What business question did your model help answer?
2. What would you recommend to Airbnb or its hosts?
3. What could you do next to improve this model or make it more useful?
4. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. This model helps answer: What listing characteristics and quality metrics have the greatest impact on Airbnb prices, and can we reliably predict a competitive price for a listing?

2. I would recommend that hosts focus heavily on physical capacity metrics (accommodates and bathrooms) and maintain high ratings on perceived value. For Airbnb, I would recommend creating smart pricing tools that dynamically adjust prices based on geographical coordinates and value ratings.

3. To make the model more useful, we should integrate categorical columns such as room_type (entire home vs. private room) and instant_bookable more effectively, handle potential outliers in price, and explore non-linear modeling algorithms.

4. This satisfies my customized learning goal of applying regression metrics to draw actionable business strategies from real-world datasets.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [21]:
!jupyter nbconvert --to html "assignment_11_regression.ipynb"

[NbConvertApp] Converting notebook assignment_11_regression.ipynb to html
[NbConvertApp] Writing 367296 bytes to assignment_11_regression.html
